# 04d — Features temporales (10 features)


In [1]:
import pandas as pd
import numpy as np
import re, time
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'data_raw'
DATA_QC_GUSTOS = ROOT / 'data' / 'data_qc_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '02_features'
DATA_QC_GUSTOS.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')

OID_RE = re.compile(r'[0-9a-f]{24}')
def clean_id(s):
    if pd.isna(s): return None
    m = OID_RE.search(str(s))
    return m.group(0) if m else None

sample = pd.read_parquet(DATA_QC_GUSTOS / 'sample_user_ids_gustos.parquet')
sample_ids = set(sample['user_id'])
N = len(sample)
features = pd.DataFrame({'user_id': sample['user_id'].values})
print(f'Sample N: {N:,}')

t0 = time.time()


Sample N: 114,412


In [2]:
# Mapping char→user (necesario para fights/arena)
chars = pd.read_csv(DATA_RAW/'characters.csv', usecols=['_id','user_id'], low_memory=False)
chars['char_id'] = chars['_id'].apply(clean_id)
chars['user_id'] = chars['user_id'].apply(clean_id)
chars = chars.dropna(subset=['char_id','user_id'])
char_to_user = dict(zip(chars['char_id'], chars['user_id']))
del chars
print(f'char_to_user: {len(char_to_user):,}')


char_to_user: 1,336,143


In [3]:
# T01-T02: claim consistency desde snapshot
# user_daily_rewards solo tiene snapshot. Usamos last_claimed_reward_day / current_day como proxy.
# Limitación: no podemos calcular ventana sliding 30d/60d real.
print('[rewards consistency]...')
t = time.time()
rw = pd.read_csv(DATA_RAW/'user_daily_rewards.csv',
                 usecols=['user_id','current_day','last_claimed_reward_day','sets_completed'], low_memory=False)
rw['user_id'] = rw['user_id'].apply(clean_id)
rw = rw[rw['user_id'].isin(sample_ids)]
rw_agg = rw.groupby('user_id').agg({'current_day':'max','last_claimed_reward_day':'max','sets_completed':'max'})
# Proxy: tasa de claims completados sobre días posibles
rw_agg['claim_consistency_30d_proxy'] = (rw_agg['last_claimed_reward_day'].clip(upper=30) / 30).clip(upper=1.0)
rw_agg['claim_consistency_60d_proxy'] = ((rw_agg['sets_completed'].fillna(0)*30 + rw_agg['last_claimed_reward_day'].fillna(0)).clip(upper=60) / 60).clip(upper=1.0)
features['claim_consistency_30d'] = features['user_id'].map(rw_agg['claim_consistency_30d_proxy'].to_dict())
features['claim_consistency_60d'] = features['user_id'].map(rw_agg['claim_consistency_60d_proxy'].to_dict())
del rw, rw_agg
print(f'OK {time.time()-t:.1f}s')


[rewards consistency]...


OK 1.2s


In [4]:
# T03 currency: pct_days_active_30d + binge + weekend_pct
print('[currency temporal]... chunks')
t = time.time()
days_active = {}    # user -> set of date
per_day_counts = {} # user -> day -> count
weekend_n = {}; total_n = {}
for chunk in pd.read_csv(DATA_RAW/'currency_transactions.csv',
                         usecols=['user_id','created_at'], chunksize=1_000_000, low_memory=False):
    chunk['user_id'] = chunk['user_id'].apply(clean_id)
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    chunk['ts'] = pd.to_datetime(chunk['created_at'], errors='coerce', utc=True)
    chunk = chunk.dropna(subset=['ts'])
    chunk['day'] = chunk['ts'].dt.date
    chunk['is_weekend'] = chunk['ts'].dt.dayofweek >= 5
    for u, g in chunk.groupby('user_id'):
        if u not in days_active: days_active[u] = set()
        days_active[u].update(g['day'].unique())
        weekend_n[u] = weekend_n.get(u,0) + int(g['is_weekend'].sum())
        total_n[u] = total_n.get(u,0) + len(g)
        if u not in per_day_counts: per_day_counts[u] = {}
        for d, n in g.groupby('day').size().items():
            per_day_counts[u][d] = per_day_counts[u].get(d,0) + n

features['pct_days_active_currency_30d'] = features['user_id'].map({u: min(len(v),30)/30 for u,v in days_active.items()})
features['weekend_pct_currency'] = features['user_id'].map({u: weekend_n.get(u,0)/v for u,v in total_n.items()})
def binge(d):
    if not d: return np.nan
    vals = list(d.values()); m = np.median(vals)
    return max(vals)/m if m>0 else np.nan
features['binge_index_currency'] = features['user_id'].map({u: binge(d) for u,d in per_day_counts.items()})
del days_active, per_day_counts, weekend_n, total_n
print(f'OK {time.time()-t:.1f}s')


[currency temporal]... chunks


OK 13.1s


In [5]:
# T04 fights: pct_days_active_fights_30d + binge + weekend_pct
print('[fights temporal]... chunks')
t = time.time()
days_active_f = {}; per_day_f = {}; weekend_f = {}; total_f = {}
for chunk in pd.read_csv(DATA_RAW/'fights_log.csv',
                         usecols=['player_id','start_time'], chunksize=500_000, low_memory=False):
    chunk['player_id'] = chunk['player_id'].apply(clean_id)
    chunk['user_id'] = chunk['player_id'].map(char_to_user)
    chunk = chunk.dropna(subset=['user_id'])
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    chunk['ts'] = pd.to_datetime(chunk['start_time'], unit='s', utc=True, errors='coerce')
    chunk = chunk.dropna(subset=['ts'])
    chunk['day'] = chunk['ts'].dt.date
    chunk['is_weekend'] = chunk['ts'].dt.dayofweek >= 5
    for u, g in chunk.groupby('user_id'):
        if u not in days_active_f: days_active_f[u] = set()
        days_active_f[u].update(g['day'].unique())
        weekend_f[u] = weekend_f.get(u,0) + int(g['is_weekend'].sum())
        total_f[u] = total_f.get(u,0) + len(g)
        if u not in per_day_f: per_day_f[u] = {}
        for d, n in g.groupby('day').size().items():
            per_day_f[u][d] = per_day_f[u].get(d,0) + n

features['pct_days_active_fights_30d'] = features['user_id'].map({u: min(len(v),30)/30 for u,v in days_active_f.items()})
features['weekend_pct_fights'] = features['user_id'].map({u: weekend_f.get(u,0)/v for u,v in total_f.items()})
def binge(d):
    if not d: return np.nan
    vals = list(d.values()); m = np.median(vals)
    return max(vals)/m if m>0 else np.nan
features['binge_index_fights'] = features['user_id'].map({u: binge(d) for u,d in per_day_f.items()})
del days_active_f, per_day_f, weekend_f, total_f
print(f'OK {time.time()-t:.1f}s')


[fights temporal]... chunks


OK 60.7s


In [6]:
# T05 coll_acquisition_velocity_60d + items_creation_velocity_30d
print('[items velocity]...')
t = time.time()
# Cargar collection con fechas
coll = pd.read_csv(DATA_RAW/'user_items_collection.csv', usecols=['user_id','created_at'], low_memory=False)
coll['user_id'] = coll['user_id'].apply(clean_id)
coll = coll[coll['user_id'].isin(sample_ids)]
coll['ts'] = pd.to_datetime(coll['created_at'], errors='coerce', utc=True)
coll_60 = coll[coll['ts']>=REFERENCE_DATE-pd.Timedelta(days=60)].groupby('user_id').size()
coll_total = coll.groupby('user_id').size()
features['coll_acquisition_velocity_60d'] = features['user_id'].map((coll_60/coll_total).to_dict()).fillna(0)
del coll

# items creation velocity 30d
items_30 = {}
for chunk in pd.read_csv(DATA_RAW/'user_items.csv', usecols=['user_id','created_at'], chunksize=2_000_000, low_memory=False):
    chunk['user_id'] = chunk['user_id'].apply(clean_id)
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    chunk['ts'] = pd.to_datetime(chunk['created_at'], errors='coerce', utc=True)
    recent = chunk[chunk['ts']>=REFERENCE_DATE-pd.Timedelta(days=30)]
    for u, n in recent.groupby('user_id').size().items():
        items_30[u] = items_30.get(u,0) + n
features['items_creation_velocity_30d'] = features['user_id'].map(items_30).fillna(0)
print(f'OK {time.time()-t:.1f}s')


[items velocity]...


OK 37.0s


In [7]:
assert len(features)==N and features['user_id'].is_unique
features.to_parquet(DATA_QC_GUSTOS / 'features_temporales.parquet', index=False)
print(f'Guardado: {features.shape}')
nan_rates = (features.isna().mean()*100).round(2)
print(nan_rates.sort_values(ascending=False).head(10).to_string())
rep_dir = INFORMES / '04d_temporales'
rep_dir.mkdir(parents=True, exist_ok=True)
report = [f'# 04d - Temporales\n', f'**Shape**: {features.shape}\n', f'**Tiempo**: {time.time()-t0:.1f}s\n', '\n| Feature | %NaN |', '|---|---:|']
for f, v in nan_rates.sort_values(ascending=False).items():
    report.append(f'| {f} | {v} |')
(rep_dir / 'execution_report.md').write_text('\n'.join(report))


Guardado: (114412, 11)
pct_days_active_fights_30d       82.01
weekend_pct_fights               82.01
binge_index_fights               82.01
pct_days_active_currency_30d     81.34
weekend_pct_currency             81.34
binge_index_currency             81.34
claim_consistency_30d             0.49
claim_consistency_60d             0.49
user_id                           0.00
coll_acquisition_velocity_60d     0.00


465